In [5]:
import os
import random
import shutil
from glob import glob
import math

data_dir = "D:/F25-156/ISLES24/processed"
dwi_dir = os.path.join(data_dir, "images")
mask_dir = os.path.join(data_dir, "masks")


out_root = "D:/F25-156/ISLES24 Split"

In [6]:
# Collect files
dwi_files = sorted(glob(os.path.join(dwi_dir, "*.nii")))
mask_files = sorted(glob(os.path.join(mask_dir, "*.nii")))

print(f"DWI samples: {len(dwi_files)}, Masks: {len(mask_files)}")

assert len(dwi_files) == len(mask_files), "Mismatch in DWI and mask count"

pairs = list(zip(dwi_files, mask_files))
random.seed(42)
random.shuffle(pairs)

N = len(pairs)
print(f"📦 Total samples available: {N}")

DWI samples: 149, Masks: 149
📦 Total samples available: 149


In [7]:
# ---- Split sizes ----
train_total = 100
val_total = 20
test_total = 29

assert train_total + val_total + test_total <= len(pairs), "Not enough samples"

train_pairs = pairs[:train_total]
val_pairs = pairs[train_total:train_total + val_total]
test_pairs = pairs[train_total + val_total:train_total + val_total + test_total]

# ---- Split train into 2 clients (50 each) ----
num_clients = 2
samples_per_client = train_total // num_clients

client_pairs = [
    train_pairs[i * samples_per_client:(i + 1) * samples_per_client]
    for i in range(num_clients)
]

# ---- Helper function to create dirs ----
def make_dirs(path):
    os.makedirs(path, exist_ok=True)

# ---- Create folder structure ----
for c in range(num_clients):
    make_dirs(os.path.join(out_root, f"train/client_{c}/DWI"))
    make_dirs(os.path.join(out_root, f"train/client_{c}/masks"))

make_dirs(os.path.join(out_root, "val/DWI"))
make_dirs(os.path.join(out_root, "val/masks"))
make_dirs(os.path.join(out_root, "test/DWI"))
make_dirs(os.path.join(out_root, "test/masks"))

# ---- Copy files ----
def copy_pairs(pairs, dwi_out, mask_out):
    for dwi, mask in pairs:
        shutil.copy(dwi, os.path.join(dwi_out, os.path.basename(dwi)))
        shutil.copy(mask, os.path.join(mask_out, os.path.basename(mask)))

# Train (clients)
for i, cp in enumerate(client_pairs):
    copy_pairs(
        cp,
        os.path.join(out_root, f"train/client_{i}/DWI"),
        os.path.join(out_root, f"train/client_{i}/masks"),
    )

# Validation
copy_pairs(
    val_pairs,
    os.path.join(out_root, "val/DWI"),
    os.path.join(out_root, "val/masks"),
)

# Test
copy_pairs(
    test_pairs,
    os.path.join(out_root, "test/DWI"),
    os.path.join(out_root, "test/masks"),
)

# ---- Summary ----
print("✅ Data split completed")
print(f"Train total: {len(train_pairs)} (20 per client)")
print(f"Validation: {len(val_pairs)}")
print(f"Test: {len(test_pairs)}")


✅ Data split completed
Train total: 100 (20 per client)
Validation: 20
Test: 29
